# Reproduce Fig.2 Part 1 - GEM 1st-pass
Set up all configurations

In [1]:
## CONFIG
GH_NODES = 200
MAX_ITER = 200
MU_GAMMA = 0.1
MU_KAPPA = -0.01
KAPPA_MAX=-0.00001

OPTIMAL_KAPPA_INIT = True 

RIDGE_KAPPA = 0.1
RIDGE_GAMMA = 0.1
RIDGE_B = 0.1
RIDGE_A = 0.5
RIDGE_X0 = 1000

strict_Y_int = False

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import joblib

# Graphical configs
# 1. Reset seaborn to baseline
sns.set_style('whitegrid')
sns.set_context('paper') # Remove font_scale!

# 2. Hardcode the absolute font sizes for publication
plt.rcParams.update({
    'font.size': 9,             # Global baseline
    'figure.titlesize': 9,      # Global figure titles (Added)
    'axes.titlesize': 10,       # Title of the subplot (ax.set_title)
    'axes.labelsize': 9,        # X and Y axis labels
    'xtick.labelsize': 8,       # Numbers on the X axis
    'ytick.labelsize': 8,       # Numbers on the Y axis
    'legend.fontsize': 8,       # Legend text
    'legend.title_fontsize': 8  # Legend title
})

### Data loading

In [ ]:
B_coding = pd.read_csv('data/B_coding.csv', index_col=0)
B_language = pd.read_csv('data/B_language.csv', index_col=0)
B_reasoning = pd.read_csv('data/B_reasoning.csv', index_col=0)
B_math = pd.read_csv('data/B_math.csv', index_col=0)

Y_coding = pd.read_csv('data/Y_coding.csv', index_col=0)
Y_language = pd.read_csv('data/Y_language.csv', index_col=0)
Y_reasoning = pd.read_csv('data/Y_reasoning.csv', index_col=0)
Y_math = pd.read_csv('data/Y_math.csv', index_col=0)

c_coding = pd.read_csv('data/c_coding.csv', index_col=0).set_index('question_id')
c_language = pd.read_csv('data/c_language.csv', index_col=0).set_index('question_id')
c_reasoning = pd.read_csv('data/c_reasoning.csv', index_col=0).set_index('question_id')
c_math = pd.read_csv('data/c_math.csv', index_col=0).set_index('question_id')

In [ ]:
B = pd.concat([B_language, B_coding, B_reasoning, B_math],axis=0)
Y = pd.concat([Y_language, Y_coding, Y_reasoning, Y_math],axis=0)
c = pd.concat([c_language, c_coding, c_reasoning, c_math],axis=0)

In [ ]:
from reasoning_aware_irt.livebench_utils import reindex
_ = reindex(Y, B, c) # little helper to make sure that all dataframes have exactly the same sorted question_id indices

In [ ]:
Y_ij = Y.values
B_ij = B.values.astype(int)
c_i = c.values.squeeze()

print(Y_ij.shape, B_ij.shape)
print(Y_ij.mean(), B_ij.mean())
print(c_i.shape, c_i[:5])

In [ ]:
model_list = np.array(list(B.columns)) 
print(model_list.shape, model_list[-5:])

In [ ]:
x0_j = np.log1p(B_ij.mean(axis=0)) # budget regularizer (Eq.12 in the paper)

# Evaluate with Reasoning-Aware 3PL's GEM

In [ ]:
from reasoning_aware_irt.gem_quad import CalibrationData, CalibrationConfig
from reasoning_aware_irt.gem_quad import ReasoningAware3PLQuadraticCalibrator as ReasoningAware3PLCalibrator

data = CalibrationData(Y_ij=Y_ij, B_ij=B_ij, c_i=c_i)
cfg = CalibrationConfig()
cfg # lets see the default config


In [ ]:
%%time
init_param_dict = None
    
cfg = CalibrationConfig(
gh_nodes=GH_NODES, max_iter=MAX_ITER, verbose=True,
ridge_kappa=RIDGE_KAPPA, ridge_gamma=RIDGE_GAMMA, 
ridge_a=RIDGE_A, mu_gamma=MU_GAMMA, ridge_b=RIDGE_B,
mu_kappa=MU_KAPPA, kappa_max=KAPPA_MAX, tol=0.001,
ridge_x0=RIDGE_X0, x0_j=x0_j, optimal_kappa=OPTIMAL_KAPPA_INIT)

calibrator = ReasoningAware3PLCalibrator(cfg)
print(cfg)

fit_results = calibrator.fit(data, strict_Y_int=strict_Y_int, init_param_dict=init_param_dict)

In [ ]:
fit_results.info['c_i'] = c_i

# Explore Output

In [ ]:
from operator import itemgetter
def print_model_summary(model_list, fit_results, Y_ij, sort_by="m", ascending=False):
    data = []
    for j, m in enumerate(model_list):
        d={} # temp dict
        d['b_theta'] = fit_results.theta_j_mean[j]
        d['theta_se'] = fit_results.theta_j_se[j]
        d['gamma'] = fit_results.gamma_j[j]
        d['gamma_se'] = fit_results.gamma_bracket_half_tau[j]

        d['kappa'] = np.clip(fit_results.kappa_j[j], a_min=-1.00, a_max=KAPPA_MAX)
        d['kappa_se'] = fit_results.kappa_bracket_half_tau[j] # fit_results.kappa_bracket_half_ideal[j]

        gse, kse = d['gamma_se'], d['kappa_se'] # fit_results.gamma_bracket_half_ideal[j] , fit_results.kappa_bracket_half_tau[j]# 
        N_SE = 0.0
        MIN_BUDGET, MAX_BUDGET = 32, 16384
        d['optimal_log_budget'] = -(d['gamma']+N_SE*gse)/(2*(d['kappa']+N_SE*kse))   # conservative with N_SE
        d['optimal_budget'] = np.clip(np.exp(d['optimal_log_budget']) - 1, MIN_BUDGET, MAX_BUDGET)

        d["optimal_log_budget"] = np.log1p(d["optimal_budget"]) # also clip here
        
        d['avg_budget'] = B_ij[:,j].mean()
        x_j = np.log1p(d['avg_budget'])
        d['eff_theta'] = d['b_theta'] + x_j* d['gamma'] + x_j**2 * d['kappa']
        d['best_theta'] = d['b_theta'] + d['optimal_log_budget'] * d['gamma'] + d['optimal_log_budget']**2 * d['kappa']
        
        d['eff_se'] = d['theta_se'] + x_j* gse
        d['best_se'] = d['theta_se'] + d['optimal_log_budget']* gse
        
        d['avg_y'] = Y_ij[:, j].mean()
        d['m'] = m
        data.append(d)

    data.sort(key=itemgetter(sort_by), reverse=not ascending)

    for d in data:
        # se is quite small, so we ignore them
        print(
            f"{d['m']:<{35}} | "
            f"BaseTheta: {d['b_theta']:6.2f} ± {d['theta_se']:.3f}| "
            f"Gamma: {d['gamma']:7.4f} ± {d['gamma_se']:.4f}| "
            f"Kappa: {d['kappa']:7.4f} ± {d['kappa_se']:.4f} |"
            f"Avg Y : {d['avg_y']:7.3f}"
        )
        print(
            f"Best Budget: {d['optimal_budget']:5.0f}|"
            f"BestTheta: {d['best_theta']:7.3f} ± {d['best_se']:7.3f}|"
            f"Avg. EffTheta: {d['eff_theta']:7.3f} ± {d['eff_se']:7.3f}|"
        )
        print()
    return data
data2 = print_model_summary(model_list, fit_results, Y_ij, sort_by="best_theta")

## Post-hoc Refinement (Appendix B)

In [ ]:
from reasoning_aware_irt.posthoc import plot_sliced_bins

In [ ]:
%%time
best_B_j_list = []
best_latent_list = []
gamma_list = []
kappa_list = []

for j in range(len(model_list)):
    ds2 = {
        'x': np.vstack([fit_results.b_i, B_ij[:,j]]).T,
        'y': Y_ij[:, j],
        'title': f'{model_list[j]}',
        #'plot_function': plot_bins_2d,
        'plot_function_argv' : {
            'min_points_per_bin':3, 
            'use_quantiles':True,
            'x1_label':"b_i",
            'x2_label':"B_ij",
            'y_label':"Average Y",
            'fit_object':fit_results, # valid only if slice_axis is B_ij
            }
        }
    ret_dict = plot_sliced_bins(ds2['x'], ds2['y'], slice_axis=1, num_slices=8, num_bins_per_slice=10, title=ds2['title'],
                     **ds2['plot_function_argv'])
    best_B_j_list.append(ret_dict['best_B_j'])
    best_latent_list.append(ret_dict['best_latent'])
    gamma_list.append(ret_dict['gamma'])
    kappa_list.append(ret_dict['kappa'])

In [ ]:
posthoc_first_pass_dict = {}
posthoc_first_pass_dict['best_B_j_list'] = best_B_j_list
posthoc_first_pass_dict['best_latent_list'] = best_latent_list
posthoc_first_pass_dict['gamma_list'] = gamma_list
posthoc_first_pass_dict['kappa_list'] = kappa_list

In [ ]:
joblib.dump(posthoc_first_pass_dict, 'data/posthoc_first_pass_dict.pkl')
joblib.dump(fit_results, 'data/first_pass_fit_results.pkl')